# $\color{cyan}{\text{Imports and Setup}}$

## $\color{yellow}{\text{Imports}}$

In [13]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import curve_fit

## $\color{yellow}{\text{Setup}}$

In [16]:
def four_pl(x, bottom, top, ec50, hill):
    '''Evaluates the four-parameter logistic model for standard-curve fitting.

    Args:
        x (array-like): Concentration values.
        bottom, top, ec50, hill (float): Logistic-model coefficients.

    Returns:
        array-like: Predicted signal values.
    '''

    # Evaluates and returns the four-parameter logistic model prediction
    return top + (bottom - top) / (1 + (x / ec50)**(hill))

In [23]:
def inverse_five_pl(y, a, d, c, b, e):
    
    try:
        ratio = (a - d) / (y - d)
        if ratio < 0:
            return np.nan
        
        inner = (ratio ** (1 / e)) - 1
        if inner < 0:
            return 0.0 
        
        return c * (inner ** (1 / b))
    
    except Exception:
        return np.nan

In [24]:
def inverse_four_pl(y, a, d, c, b):
    
    return inverse_five_pl(y, a, d, c, b, 1.0)

In [32]:
def fit_curve(data):
    '''Fits candidate logistic models and selects the best standard-curve result.

    Args:
        data (pd.DataFrame): Concentration and response measurements.

    Returns:
        tuple: Fitted curve values and the optimized parameters.
    '''

    # Extracts the concentration and response arrays from the data
    x, y = data['x'], data['y']

    # Generates logarithmically spaced concentration points for the fitted curve
    xfit = np.logspace(np.log10(0.5), np.log10(2500), 1000)

    # Defines the parameter bounds for the four-parameter curve fit
    bounds = ([-np.inf, -np.inf, min(x)/100, -10], [np.inf, np.inf, max(x)*100, 10])

    # Establishes the initial parameter guesses for the four-parameter curve fit
    p0 = [min(y), max(y), np.median(x), 1.0]

    # Executes the curve fitting algorithm to find optimal four-parameter coefficients
    popt, _ = curve_fit(four_pl, x, y, p0=p0, bounds=bounds, maxfev=50000)

    # Calculates the predicted response values using the optimised four-parameter model
    yfit = four_pl(xfit, *popt)
        
    # Returns the generated curve coordinates alongside the optimal parameters
    return xfit, yfit, popt

# $\color{cyan}{\text{Data and Run}}$

In [33]:
sc_data = {
    'sc1': {
        'x': [5, 10, 50, 100, 200, 1000, 1500],
        'y': [0.04, 0.09, 0.42, 0.73, 1.01, 1.29, 1.35]
    },
    'sc2': {
            'x': [5, 10, 50, 100, 200, 1000, 1500],
            'y': [0.03, 0.08, 0.41, 0.69, 0.95, 1.22, 1.27]
    },
    'sc3': {
            'x': [5, 10, 50, 100, 200, 1000, 1500],
            'y': [0.02, 0.06, 0.36, 0.71, 0.93, 1.23, 1.27]
    }
}

signal = [0.02, 0.06, 0.36, 0.71, 0.93, 1.23, 1.27]

In [35]:
# Initialises a standard Plotly figure
fig = go.Figure()

# Initialises a dictionary to store the interpolated concentration values
table_data = {'Target Signal': signal}

# Loops through each standard curve in the provided dataset
for sc_key, data in sc_data.items():
    
    # Extracts the raw data and fits the 4PL curve
    xfit, yfit, popt = fit_curve(data)

    # Adds the raw measured coordinates as a scatter plot trace
    fig.add_trace(go.Scatter(x=data['x'], y=data['y'], mode='markers', name=f'{sc_key} Raw Data'))

    # Adds the calculated optimal curve fit as a line trace
    fig.add_trace(go.Scatter(x=xfit, y=yfit, mode='lines', name=f'{sc_key} 4PL Fit'))

    # Calculates the interpolated concentration for each target signal value on this curve
    interpolated_x = [inverse_four_pl(s, *popt) for s in signal]

    # Formats the interpolated values to 2 decimal places for the table, handling NaNs
    table_data[sc_key] = [f"{val:.2f}" if not np.isnan(val) else "Out of bounds" for val in interpolated_x]

    # Filters out NaN values to accurately plot intersection points on the graph
    valid_x = [x for x in interpolated_x if not np.isnan(x)]
    valid_y = [s for s, x in zip(signal, interpolated_x) if not np.isnan(x)]

    # Adds 'X' markers at the exact points where the signals intersect the standard curve
    fig.add_trace(go.Scatter(x=valid_x, y=valid_y, mode='markers', marker=dict(symbol='x', size=8, color='black'), showlegend=False, hoverinfo='skip'))

# Loops through each target signal value to draw horizontal reference lines
for s in signal:
    
    # Adds a dashed horizontal line spanning the entire plot for the target signal
    fig.add_hline(y=s, line_dash="dash", line_color="gray", opacity=0.4)

# Updates the layout to set a logarithmic scale, dimensions, and axis titles
fig.update_layout(
    title_text="Standard Curves (4PL Fit) with Interpolated Signals",
    height=600,
    xaxis_type="log",
    xaxis_title="Concentration (ug/ml)",
    yaxis_title="Signal Response",
    template="plotly_white"
)

# Converts the collected interpolation data dictionary into a Pandas DataFrame
df_table = pd.DataFrame(table_data)

# Renders the interactive Plotly graph
fig.show()

# Displays the dataframe
display(df_table)

,Target Signal,sc1,sc2,sc3
0,0.02,0.66,3.24,5.03
1,0.06,7.12,8.25,10.67
2,0.36,41.55,42.74,46.89
3,0.71,96.87,105.57,108.52
4,0.93,161.71,189.43,187.66
5,1.23,481.45,938.05,890.03
6,1.27,643.26,2145.60,2294.50
